# 01 — Run Experiment

Runs LLM API calls and saves results to Google Drive.

**First time?** See README.md for setup instructions.

## 1. Clone Repo & Mount Drive

# =============================================================
# 01_run_experiment_v2.py
# Updated for: economics-only arguments, new authority labels,
#              new prompt template (no system prompt)
# =============================================================
# All output files are prefixed with "v2_" to clearly separate
# from earlier data collection rounds.
# =============================================================

In [7]:
# ── CELL 1: Clone Repo & Mount Drive ─────────────────────────

import os

os.chdir('/content')

# Clone repo (fresh each time)
!rm -rf /content/authority-bias-llm
!git clone https://github.com/auertobias/authority-bias-llm.git /content/authority-bias-llm

os.chdir('/content/authority-bias-llm')

from google.colab import drive
drive.mount('/content/drive')

!pip install -q google-genai openai anthropic


Cloning into '/content/authority-bias-llm'...
remote: Enumerating objects: 195, done.
remote: Counting objects: 100% (195/195), done.
remote: Compressing objects: 100% (183/183), done.
remote: Total 195 (delta 119), reused 23 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (195/195), 1.41 MiB | 8.18 MiB/s, done.
Resolving deltas: 100% (119/119), done.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Configure

In [8]:
# ── CELL 2: Configure ────────────────────────────────────────
import os, sys, time
sys.path.insert(0, '.')

from src.config import DATA_PATH, RESULTS_PATH, N_REPS, CHECKPOINT_EVERY

os.makedirs(DATA_PATH, exist_ok=True)
os.makedirs(RESULTS_PATH, exist_ok=True)
print(f"Data saves to: {DATA_PATH}")

VERSION = "v3"

# Pin model versions (written into every output row)
MODEL_VERSIONS = {
    'gpt':      'gpt-4o-mini-2024-07-18',
    'deepseek': 'deepseek-chat',
    'gemini':   'gemini-2.5-flash',
    'claude':   'claude-haiku-4-5-20251001',
    'llama':    'meta-llama/Llama-3.3-70B-Instruct-Turbo',
}

# Per-model pause (seconds between API calls)
PAUSE_BY_MODEL = {
    'gpt':      0.0,
    'deepseek': 0.0,
    'gemini':   0.3,
    'claude':   1.0,
    'llama':    0.0,
}

# API Keys
from google.colab import userdata
GEMINI_API_KEY    = userdata.get('GEMINI_API_KEY')
DEEPSEEK_API_KEY  = userdata.get('DEEPSEEK_API_KEY')
OPENAI_API_KEY    = userdata.get('OPEN_API_KEY')
ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
TOGETHER_API_KEY  = userdata.get('TOGETHER_API_KEY')

# Load stimuli
import pandas as pd
arguments   = pd.read_csv(DATA_PATH + "arguments_economics.csv")
authorities = pd.read_csv(DATA_PATH + "authority_labels.csv", sep=';')

# Pre-flight assertions
assert len(arguments)   == 200, f"Expected 200 arguments, got {len(arguments)}"
assert len(authorities) == 24,  f"Expected 24 authority labels, got {len(authorities)}"

print(f"Arguments:   {len(arguments)} rows, {arguments['claim_id'].nunique()} claims")
print(f"Authorities: {len(authorities)} labels")
print(f"  Matched:      {len(authorities[authorities['layer']=='matched'])}")
print(f"  Reference:    {len(authorities[authorities['layer']=='reference'])}")
print(f"  Naturalistic: {len(authorities[authorities['layer']=='naturalistic'])}")
print(f"  Baseline:     {len(authorities[authorities['layer']=='baseline'])}")

import subprocess
print(f"\nGit commit: {subprocess.check_output(['git','rev-parse','--short','HEAD']).decode().strip()}")
print(f"N_REPS = {N_REPS}")

Data saves to: /content/drive/MyDrive/PhD/2PhD 1Paper/data/
Arguments:   200 rows, 100 claims
Authorities: 24 labels
  Matched:      12
  Reference:    6
  Naturalistic: 4
  Baseline:     2

Git commit: e102ff6
N_REPS = 1


## 3. Load Data & Build Trials

In [9]:
trials = []
trial_id = 0

for _, arg in arguments.iterrows():
    for _, auth in authorities.iterrows():
        trial_id += 1
        trials.append({
            'trial_id':        trial_id,
            'argument_id':     arg['id'],
            'claim_id':        arg['claim_id'],
            'claim':           arg['claim'],
            'argument':        arg['argument'],
            'stance':          arg['stance'],
            'arg_type':        arg['type'],
            'topic':           arg['topic'],
            'authority_id':    auth['authority_id'],
            'authority_label': auth['label'].strip(),
            'status':          auth['status'],
            'expertise':       auth['expertise'],
            'layer':           auth['layer'],
            'pair_id':         auth['pair_id'] if pd.notna(auth.get('pair_id')) else '',
        })

trials_df = pd.DataFrame(trials)
trials_hidden = trials_df.copy()

print(f"Total trials per rep: {len(trials_df)}")
print(f"  = {len(arguments)} arguments × {len(authorities)} labels")

print(f"\n── Matched pairs ──")
matched = trials_df[trials_df['layer'] == 'matched']
print(matched.groupby(['status', 'expertise', 'arg_type']).size().unstack(fill_value=0))

print(f"\n── Reference conditions ──")
print(trials_df[trials_df['layer'] == 'reference']
      .groupby(['authority_label', 'arg_type']).size().unstack(fill_value=0))

print(f"\n── Naturalistic titles ──")
print(trials_df[trials_df['layer'] == 'naturalistic']
      .groupby(['authority_label', 'status']).size())

print(f"\n── Baseline conditions ──")
print(trials_df[trials_df['layer'] == 'baseline']
      .groupby('authority_label').size())

print(f"\nOpen trials:   {len(trials_df)}")
print(f"Hidden trials: {len(trials_hidden)}")

Total trials per rep: 4800
  = 200 arguments × 24 labels

── Matched pairs ──
arg_type           descriptive  normative
status expertise                         
high   irrelevant          208        192
       relevant            208        192
low    irrelevant          208        192
       relevant            208        192
medium irrelevant          208        192
       relevant            208        192

── Reference conditions ──
arg_type         descriptive  normative
authority_label                        
ChatGPT                  104         96
Claude                   104         96
Deepseek                 104         96
Gemini                   104         96
Llama AI                 104         96
an AI model              104         96

── Naturalistic titles ──
authority_label               status
a federal court judge         high      200
a four-star military general  high      200
a military recruit            low       200
a paralegal                   low       20

In [10]:
import importlib
import src.prompts
importlib.reload(src.prompts)
from src.prompts import build_prompt, build_prompt_hidden

print("── OPEN condition openings ──\n")
for _, auth in authorities.iterrows():
    fake_row = {
        'claim': '[CLAIM]',
        'authority_label': auth['label'],
        'argument': '[ARGUMENT]',
        'stance': 'support',
    }
    lines = [ln for ln in build_prompt(fake_row).split('\n') if ln.strip()]
    print(f"  {auth['label']:45s}  →  {lines[2]}")

print("\n── HIDDEN condition openings ──\n")
for _, auth in authorities.iterrows():
    fake_row = {
        'claim': '[CLAIM]',
        'authority_label': auth['label'],
        'stance': 'support',
    }
    lines = [ln for ln in build_prompt_hidden(fake_row).split('\n') if ln.strip()]
    print(f"  {auth['label']:45s}  →  {lines[2].split(',')[0]}")

── OPEN condition openings ──

  a professor of economics                       →  A professor of economics responded:
  a professor of medicine                        →  A professor of medicine responded:
  a Nobel laureate in economics                  →  A Nobel laureate in economics responded:
  a Nobel laureate in physics                    →  A Nobel laureate in physics responded:
  a PhD student in economics                     →  A PhD student in economics responded:
  a PhD student in law                           →  A PhD student in law responded:
  a lecturer in economics                        →  A lecturer in economics responded:
  a lecturer in philosophy                       →  A lecturer in philosophy responded:
  a research assistant in economics              →  A research assistant in economics responded:
  a research assistant in neuroscience           →  A research assistant in neuroscience responded:
  a master's student in economics                →  A master's s

In [11]:
# ── Hidden condition: FULL design (all 26 labels × 200 arguments) ──
trials_hidden = trials_df.copy().reset_index(drop=True)

print(f"Open trials:   {len(trials_df)}")
print(f"Hidden trials: {len(trials_hidden)}")

Open trials:   4800
Hidden trials: 4800


## 4. Choose Models & Run

In [13]:
import importlib
import src.models, src.parsing
importlib.reload(src.models)
importlib.reload(src.parsing)

from src.models import make_gemini_fn, make_gpt_fn, make_deepseek_fn, make_claude_fn, make_llama_fn
from src.parsing import extract_rating, extract_analysis

# Retry wrapper
def with_retry(fn, max_retries=4, base_delay=2):
    def wrapped(prompt):
        for attempt in range(max_retries):
            try:
                out = fn(prompt)
                if out is not None:
                    return out
            except Exception as e:
                print(f"    retry {attempt+1}/{max_retries}: {type(e).__name__}: {e}")
            time.sleep(base_delay * (2 ** attempt))
        return None
    return wrapped

# Uncomment ONLY the models you're running this session
MODELS = {
     'gpt':      with_retry(make_gpt_fn(OPENAI_API_KEY,        MODEL_VERSIONS['gpt'])),
    # 'deepseek': with_retry(make_deepseek_fn(DEEPSEEK_API_KEY, MODEL_VERSIONS['deepseek'])),
    # 'gemini':   with_retry(make_gemini_fn(GEMINI_API_KEY,     MODEL_VERSIONS['gemini'])),
    # 'claude':   with_retry(make_claude_fn(ANTHROPIC_API_KEY,  MODEL_VERSIONS['claude'])),
    # 'llama':    with_retry(make_llama_fn(TOGETHER_API_KEY,    MODEL_VERSIONS['llama'])),
}

print(f"Models to run: {list(MODELS.keys())}")
for m in MODELS:
    print(f"  {m}: version={MODEL_VERSIONS[m]}, pause={PAUSE_BY_MODEL[m]}s")

Models to run: ['gpt']
  gpt: version=gpt-4o-mini-2024-07-18, pause=0.0s


In [ ]:
from datetime import datetime

def run_condition(model_name, model_fn, trials_source, condition, build_fn):
    checkpoint_path = DATA_PATH + f"{VERSION}_checkpoint_{model_name}_{condition}.csv"
    pause = PAUSE_BY_MODEL[model_name]

    # Resume from checkpoint if it exists
    if os.path.exists(checkpoint_path):
        prior = pd.read_csv(checkpoint_path)
        results = prior.to_dict('records')
        done_keys = set(zip(prior['trial_id'], prior['rep']))
        print(f"  Resuming: {len(results)} trials already done, skipping those")
    else:
        results = []
        done_keys = set()

    print(f"\n{'='*60}")
    print(f"  {VERSION} | {model_name.upper()} — {condition.upper()} | N_REPS={N_REPS} | pause={pause}s")
    print(f"{'='*60}")

    for rep in range(N_REPS):
        seed = (1000 if condition == 'open' else 2000) * (rep + 1)
        run_order = trials_source.sample(frac=1, random_state=seed).reset_index(drop=True)
        print(f"\n--- Rep {rep+1}/{N_REPS} (seed={seed}, {len(run_order)} trials) ---")

        for idx, row in run_order.iterrows():
            key = (row['trial_id'], rep + 1)
            if key in done_keys:
                continue

            prompt = build_fn(row)
            raw = model_fn(prompt)
            rating = extract_rating(raw)

            results.append({
                'trial_id':        row['trial_id'],
                'rep':             rep + 1,
                'model':           model_name,
                'model_version':   MODEL_VERSIONS[model_name],
                'version':         VERSION,
                'condition':       condition,
                'argument_id':     row['argument_id'],
                'claim_id':        row['claim_id'],
                'claim':           row['claim'],
                'argument':        row['argument'] if condition == 'open' else '',
                'stance':          row['stance'],
                'arg_type':        row['arg_type'],
                'topic':           row['topic'],
                'authority_id':    row['authority_id'],
                'authority_label': row['authority_label'],
                'status':          row['status'],
                'expertise':       row['expertise'],
                'layer':           row['layer'],
                'pair_id':         row['pair_id'],
                'rating':          rating,
                'analysis':        extract_analysis(raw) if condition == 'open' else '',
                'raw_response':    raw,
            })
            done_keys.add(key)

            n_done = len(results)
            if n_done % 50 == 0:
                print(f"  {n_done} total done | last rating: {rating}")
            if n_done % CHECKPOINT_EVERY == 0:
                pd.DataFrame(results).to_csv(checkpoint_path, index=False)

            time.sleep(pause)

    # Final timestamped save
    date_str = datetime.now().strftime("%Y%m%d_%H%M")
    final_path = DATA_PATH + f"{VERSION}_data_{model_name}_{condition}_{date_str}.csv"
    df_out = pd.DataFrame(results)
    df_out.to_csv(final_path, index=False)
    valid = df_out['rating'].notna().sum()
    print(f"\n✓ Saved: {final_path}")
    print(f"  {len(df_out)} rows, {valid} valid ({100*valid/len(df_out):.1f}%)")
    return df_out


# Run all selected models, both conditions
for model_name, model_fn in MODELS.items():
    run_condition(model_name, model_fn, trials_df,    'open',   build_prompt)
    run_condition(model_name, model_fn, trials_hidden, 'hidden', build_prompt_hidden)
    print(f"\n{'='*60}")
    print(f"  DONE: {model_name.upper()} — both conditions complete")
    print(f"{'='*60}")

  Resuming: 3800 trials already done, skipping those

  v3 | DEEPSEEK — OPEN | N_REPS=1 | pause=0.0s

--- Rep 1/1 (seed=1000, 4800 trials) ---
  3850 total done | last rating: 65
  3900 total done | last rating: 55
  3950 total done | last rating: 45
  4000 total done | last rating: 35
  4050 total done | last rating: 45
  4100 total done | last rating: 15
  4150 total done | last rating: 45
  4200 total done | last rating: 65
  4250 total done | last rating: 45
  4300 total done | last rating: 65
  4350 total done | last rating: 35
  4400 total done | last rating: 65
  4450 total done | last rating: 65
  4500 total done | last rating: 35
  4550 total done | last rating: 35
  4600 total done | last rating: 65
  4650 total done | last rating: 85
  4700 total done | last rating: 35
  4750 total done | last rating: 65
  4800 total done | last rating: 65

✓ Saved: /content/drive/MyDrive/PhD/2PhD 1Paper/data/v3_data_deepseek_open_20260421_1338.csv
  4800 rows, 4800 valid (100.0%)

  v3 | DE

In [ ]:
import time
from datetime import datetime

date_str = datetime.now().strftime("%Y%m%d")

for model_name, model_fn in MODELS.items():

  # ── CONDITION 2: HIDDEN ───────────────────────────────────
    print(f"\n{'='*60}")
    print(f"  {VERSION} | {model_name.upper()} — HIDDEN condition")
    print(f"{'='*60}")

    results_hidden = []

    for rep in range(N_REPS):
        run_order = trials_hidden.sample(frac=1, random_state=rep*2000).reset_index(drop=True)
        print(f"\n--- Rep {rep+1}/{N_REPS} ({len(run_order)} trials) ---")

        for idx, row in run_order.iterrows():
            prompt = build_prompt_hidden(row)
            raw = model_fn(prompt)
            rating = extract_rating(raw)

            results_hidden.append({
                'trial_id':        row['trial_id'],
                'rep':             rep + 1,
                'model':           model_name,
                'version':         VERSION,
                'condition':       'hidden',
                'argument_id':     row['argument_id'],
                'claim_id':        row['claim_id'],
                'claim':           row['claim'],
                'argument':        '',
                'stance':          row['stance'],
                'arg_type':        row['arg_type'],
                'topic':           row['topic'],
                'authority_id':    row['authority_id'],
                'authority_label': row['authority_label'],
                'status':          row['status'],
                'expertise':       row['expertise'],
                'layer':           row['layer'],
                'pair_id':         row['pair_id'],
                'rating':          rating,
                'reasoning':       extract_reasoning(raw),
                'raw_response':    raw,
            })

            n_done = idx + 1
            if n_done % 50 == 0:
                print(f"  {n_done}/{len(run_order)} done | last rating: {rating}")

            if n_done % CHECKPOINT_EVERY == 0:
                pd.DataFrame(results_hidden).to_csv(
                    DATA_PATH + f"{VERSION}_checkpoint_{model_name}_hidden.csv",
                    index=False
                )

            time.sleep(PAUSE_SECONDS)

    # Save hidden condition
    filepath_hidden = DATA_PATH + f"{VERSION}_data_{model_name}_hidden_{date_str}.csv"
    pd.DataFrame(results_hidden).to_csv(filepath_hidden, index=False)
    valid = sum(1 for r in results_hidden if r['rating'] is not None)
    print(f"\n✓ Saved: {filepath_hidden}")
    print(f"  {len(results_hidden)} rows, {valid} valid ratings ({100*valid/len(results_hidden):.0f}%)")

    print(f"\n{'='*60}")
    print(f"  DONE: {VERSION} | {model_name.upper()} — both conditions complete")
    print(f"{'='*60}")



  v2 | LLAMA — HIDDEN condition

--- Rep 1/1 (3600 trials) ---


KeyboardInterrupt: 

## 5. Verify

In [ ]:
import glob

print("\n── v2 data files ──")
v2_files = sorted(glob.glob(DATA_PATH + f"{VERSION}_data_*.csv"))
for f in v2_files:
    df_check = pd.read_csv(f)
    valid = df_check['rating'].notna().sum()
    print(f"  {os.path.basename(f):50s} → {len(df_check)} rows, {valid} valid")

print("\n── Old v1 data files (untouched) ──")
v1_files = sorted(glob.glob(DATA_PATH + "data_*.csv"))
for f in v1_files:
    df_check = pd.read_csv(f)
    print(f"  {os.path.basename(f):50s} → {len(df_check)} rows")


── v2 data files ──
  v2_data_gemini_open_20260323.csv                   → 5200 rows, 5190 valid
  v2_data_gpt_hidden_20260317.csv                    → 4200 rows, 4199 valid
  v2_data_gpt_open_20260317_1.csv                    → 5200 rows, 5198 valid

── Old v1 data files (untouched) ──
  data_deepseek_20260312.csv                         → 1776 rows
  data_gemini_20260313.csv                           → 1776 rows
  data_gpt_20260311.csv                              → 1776 rows
  data_gpt_hidden_20260316.csv                       → 1480 rows
  data_gpt_open_20260316.csv                         → 1776 rows
